# Máquinas de Soporte Vectorial para Regresión (SVR)

## Introducción Teórica

La Regresión con Máquinas de Soporte Vectorial (Support Vector Regression - SVR) es una extensión de SVM para problemas de regresión.

In [ ]:
# IMPORTS Y CONFIGURACIÓN
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVR, LinearSVR
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, 
    mean_absolute_error, 
    r2_score
)
from sklearn.pipeline import make_pipeline
from sklearn.datasets import make_regression, fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

import mlutils
import warnings
warnings.filterwarnings("ignore")

# Configuración de estilo
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

## 1. SVR Lineal

Datos Sintéticos Lineales


In [ ]:
X_linear, y_linear = make_regression(n_samples=100, n_features=1, noise=10, random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X_linear, y_linear, alpha=0.7, s=30)
ax.set_xlabel('X', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Datos Sintéticos Lineales con Ruido', fontweight='bold', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Entrenamiento de SVR Lineal


In [ ]:
LinearSVR?

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_linear, y_linear, test_size=0.3, random_state=42
)

svr_linear = LinearSVR(C=1.0, epsilon=5, random_state=42, max_iter=10000)
svr_linear.fit(X_train, y_train)

y_train_pred = svr_linear.predict(X_train)
y_test_pred = svr_linear.predict(X_test)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
mlutils.plot_svr_results(X_train, y_train, y_train_pred, svr_linear, 
                title='SVR Lineal - Entrenamiento', ax=ax1)
mlutils.plot_svr_results(X_test, y_test, y_test_pred, svr_linear, 
                title='SVR Lineal - Prueba', ax=ax2)
plt.tight_layout()
plt.show()

print("="*60)
print("MÉTRICAS SVR LINEAL")
print("="*60)
print(f"Entrenamiento - R²: {r2_score(y_train, y_train_pred):.4f}")
print(f"Entrenamiento - MSE: {mean_squared_error(y_train, y_train_pred):.4f}")
print(f"Prueba - R²: {r2_score(y_test, y_test_pred):.4f}")
print(f"Prueba - MSE: {mean_squared_error(y_test, y_test_pred):.4f}")
print(f"Coeficientes: {svr_linear.coef_[0]:.4f}")
print(f"Intercept: {svr_linear.intercept_[0]:.4f}")

## 2. SVR con Kernel para Datos No Lineales

In [ ]:
X_nonlinear, y_nonlinear = mlutils.generate_synthetic_data(n_samples=200, noise=0.3, seed=42)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X_nonlinear, y_nonlinear, alpha=0.7, s=30)
ax.set_xlabel('X', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Datos Sintéticos No Lineales', fontweight='bold', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Comparación de Kernels


In [ ]:
kernels = ['linear', 'rbf', 'poly', 'sigmoid']
kernel_params = {
    'linear': {},
    'rbf': {'gamma': 'scale'},
    'poly': {'degree': 3, 'gamma': 'scale', 'coef0': 1},
    'sigmoid': {'gamma': 'scale', 'coef0': 0}
}

X_train, X_test, y_train, y_test = train_test_split(
    X_nonlinear, y_nonlinear, test_size=0.3, random_state=42
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

results = {}

for idx, kernel in enumerate(kernels):
    ax = axes[idx]
    
    pipeline = make_pipeline(
        StandardScaler(),
        SVR(kernel=kernel, C=1.0, epsilon=0.1, **kernel_params[kernel])
    )
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    results[kernel] = {
        'r2': r2_score(y_test, y_pred),
        'mse': mean_squared_error(y_test, y_pred)
    }
    
    mlutils.plot_svr_results(X_test, y_test, y_pred, pipeline.named_steps['svr'],
                    title=f'Kernel {kernel}\nR² = {results[kernel]["r2"]:.3f}', ax=ax)

plt.tight_layout()
plt.show()

print("="*60)
print("COMPARACIÓN DE KERNELS EN SVR")
print("="*60)
for kernel, metrics in results.items():
    print(f"{kernel:10} - R²: {metrics['r2']:.4f}, MSE: {metrics['mse']:.4f}")

## 3. Efecto de los Hiperparámetros

### Efecto del Parámetro C


In [ ]:
C_values = [0.01, 0.1, 1.0, 10.0, 100.0]

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for idx, C in enumerate(C_values):
    ax = axes[idx]
    svr = SVR(kernel='rbf', C=C, epsilon=0.1, gamma='scale')
    svr.fit(X_train, y_train)
    y_pred = svr.predict(X_test)
    mlutils.plot_svr_results(X_test, y_test, y_pred, svr, 
                    title=f'C = {C}\nR² = {r2_score(y_test, y_pred):.3f}', ax=ax)

plt.suptitle('Efecto del Parámetro C en SVR', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### Efecto del Parámetro Epsilon


In [ ]:
epsilon_values = [0.001, 0.01, 0.1, 0.5, 1.0]

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for idx, eps in enumerate(epsilon_values):
    ax = axes[idx]
    svr = SVR(kernel='rbf', C=1.0, epsilon=eps, gamma='scale')
    svr.fit(X_train, y_train)
    y_pred = svr.predict(X_test)
    mlutils.plot_svr_results(X_test, y_test, y_pred, svr, 
                    title=f'ε = {eps}\nR² = {r2_score(y_test, y_pred):.3f}', ax=ax)

plt.suptitle('Efecto del Parámetro ε en SVR', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### Efecto del Parámetro Gamma


In [ ]:
gamma_values = [0.01, 0.1, 1.0, 10.0, 100.0]

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for idx, gamma in enumerate(gamma_values):
    ax = axes[idx]
    svr = SVR(kernel='rbf', C=1.0, epsilon=0.1, gamma=gamma)
    svr.fit(X_train, y_train)
    y_pred = svr.predict(X_test)
    mlutils.plot_svr_results(X_test, y_test, y_pred, svr, 
                    title=f'γ = {gamma}\nR² = {r2_score(y_test, y_pred):.3f}', ax=ax)

plt.suptitle('Efecto del Parámetro γ en SVR (RBF)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Optimización con GridSearchCV

In [ ]:
pipeline = make_pipeline(
    StandardScaler(),
    SVR(kernel='rbf')
)

param_grid = {
    'svr__C': [0.1, 1.0, 10.0, 100.0],
    'svr__epsilon': [0.01, 0.1, 0.5, 1.0],
    'svr__gamma': ['scale', 'auto', 0.1, 1.0]
}

grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=5, 
    scoring='r2',
    n_jobs=-1, 
    verbose=1
)

print("Realizando búsqueda de hiperparámetros...")
grid_search.fit(X_train, y_train)

print("\n" + "="*60)
print("RESULTADOS DE GRIDSEARCHCV")
print("="*60)
print(f"Mejores parámetros: {grid_search.best_params_}")
print(f"Mejor puntuación CV (R²): {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
y_test_pred = best_model.predict(X_test)
test_r2 = r2_score(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

print(f"\nRendimiento en prueba:")
print(f"  R²: {test_r2:.4f}")
print(f"  MSE: {test_mse:.4f}")
print(f"  MAE: {mean_absolute_error(y_test, y_test_pred):.4f}")

fig, ax = plt.subplots(figsize=(10, 6))
mlutils.plot_svr_results(X_test, y_test, y_test_pred, best_model.named_steps['svr'],
                title=f'Mejor Modelo - R² = {test_r2:.3f}', ax=ax)
plt.tight_layout()
plt.show()

## 5. Aplicación en Dataset Real: California Housing

In [ ]:
housing = fetch_california_housing()
X_housing, y_housing = housing.data, housing.target

print("="*60)
print("DATASET: CALIFORNIA HOUSING")
print("="*60)
print(f"Número de muestras: {X_housing.shape[0]}")
print(f"Número de características: {X_housing.shape[1]}")
print(f"Características: {housing.feature_names}")
print(f"\nEstadísticas del target (precio en $100k):")
print(f"  Min: {y_housing.min():.2f}")
print(f"  Max: {y_housing.max():.2f}")
print(f"  Media: {y_housing.mean():.2f}")
print(f"  Desv. Est.: {y_housing.std():.2f}")

### Entrenamiento en California Housing


In [ ]:
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_housing, y_housing, test_size=0.2, random_state=42
)

pipeline_housing = make_pipeline(
    StandardScaler(),
    SVR(kernel='rbf', C=1.0, epsilon=0.1, gamma='scale')
)

print("Entrenando SVR en California Housing...")
pipeline_housing.fit(X_train_h, y_train_h)

y_train_pred_h = pipeline_housing.predict(X_train_h)
y_test_pred_h = pipeline_housing.predict(X_test_h)

print("\n" + "="*60)
print("RENDIMIENTO EN CALIFORNIA HOUSING")
print("="*60)
print(f"Entrenamiento:")
print(f"  R²: {r2_score(y_train_h, y_train_pred_h):.4f}")
print(f"  MSE: {mean_squared_error(y_train_h, y_train_pred_h):.4f}")
print(f"  MAE: {mean_absolute_error(y_train_h, y_train_pred_h):.4f}")
print(f"\nPrueba:")
print(f"  R²: {r2_score(y_test_h, y_test_pred_h):.4f}")
print(f"  MSE: {mean_squared_error(y_test_h, y_test_pred_h):.4f}")
print(f"  MAE: {mean_absolute_error(y_test_h, y_test_pred_h):.4f}")

### Optimización en Dataset Real


In [ ]:
param_grid_housing = {
    'svr__C': [0.1, 1.0, 10.0],
    'svr__epsilon': [0.01, 0.1, 0.5],
    'svr__gamma': ['scale', 'auto', 0.1, 1.0]
}

grid_search_housing = GridSearchCV(
    pipeline_housing,
    param_grid_housing,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

print("Optimizando SVR para California Housing...")
grid_search_housing.fit(X_train_h, y_train_h)

print("\n" + "="*60)
print("RESULTADOS OPTIMIZADOS")
print("="*60)
print(f"Mejores parámetros: {grid_search_housing.best_params_}")
print(f"Mejor CV R²: {grid_search_housing.best_score_:.4f}")

best_model_h = grid_search_housing.best_estimator_
y_test_pred_h_opt = best_model_h.predict(X_test_h)

print(f"\nRendimiento en prueba (optimizado):")
print(f"  R²: {r2_score(y_test_h, y_test_pred_h_opt):.4f}")
print(f"  MSE: {mean_squared_error(y_test_h, y_test_pred_h_opt):.4f}")
print(f"  MAE: {mean_absolute_error(y_test_h, y_test_pred_h_opt):.4f}")

### Visualización de Resultados


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: Predicciones vs Reales
ax = axes[0]
ax.scatter(y_test_h, y_test_pred_h_opt, alpha=0.5, s=20)
ax.plot([y_test_h.min(), y_test_h.max()], [y_test_h.min(), y_test_h.max()], 
        'r--', linewidth=2, label='Predicción perfecta')
ax.set_xlabel('Valor Real ($100k)', fontsize=12)
ax.set_ylabel('Valor Predicho ($100k)', fontsize=12)
ax.set_title('Predicciones vs Reales - Test', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Distribución de errores
ax = axes[1]
errors = y_test_pred_h_opt - y_test_h
ax.hist(errors, bins=30, alpha=0.7, color='blue', edgecolor='black')
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Error cero')
ax.axvline(np.mean(errors), color='green', linestyle='--', linewidth=2, 
          label=f'Media: {np.mean(errors):.3f}')
ax.axvline(np.percentile(errors, 95), color='orange', linestyle=':', linewidth=2,
          label=f'Percentil 95: {np.percentile(errors, 95):.3f}')
ax.set_xlabel('Error de Predicción', fontsize=12)
ax.set_ylabel('Frecuencia', fontsize=12)
ax.set_title('Distribución de Errores', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Comparación con Otros Modelos

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=1.0),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'SVR (RBF)': SVR(kernel='rbf', C=1.0, epsilon=0.1, gamma='scale')
}

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_nonlinear, y_nonlinear, test_size=0.3, random_state=42
)

results_models = {}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (name, model) in enumerate(models.items()):
    ax = axes[idx]
    
    pipeline = make_pipeline(StandardScaler(), model)
    pipeline.fit(X_train_c, y_train_c)
    y_pred = pipeline.predict(X_test_c)
    
    r2 = r2_score(y_test_c, y_pred)
    mse = mean_squared_error(y_test_c, y_pred)
    
    results_models[name] = {'R²': r2, 'MSE': mse}
    
    ax.scatter(X_test_c, y_test_c, alpha=0.5, s=20)
    ax.scatter(X_test_c, y_pred, alpha=0.5, s=20, color='red')
    ax.set_title(f'{name}\nR² = {r2:.3f}', fontweight='bold')
    ax.set_xlabel('X')
    ax.set_ylabel('y')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

results_df = pd.DataFrame(results_models).T
results_df = results_df.sort_values('R²', ascending=False)

print("="*60)
print("COMPARACIÓN DE MODELOS DE REGRESIÓN")
print("="*60)
print(results_df.round(4).to_string())

## 7. Validación Cruzada

In [ ]:
best_svr = grid_search.best_estimator_
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores_r2 = cross_val_score(best_svr, X_train, y_train, cv=cv, scoring='r2')
scores_mse = cross_val_score(best_svr, X_train, y_train, cv=cv, scoring='neg_mean_squared_error')
scores_mae = cross_val_score(best_svr, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error')

print("="*60)
print("VALIDACIÓN CRUZADA (5-FOLD)")
print("="*60)
print(f"R²: {scores_r2.mean():.4f} ± {scores_r2.std():.4f}")
print(f"MSE: {-scores_mse.mean():.4f} ± {scores_mse.std():.4f}")
print(f"MAE: {-scores_mae.mean():.4f} ± {scores_mae.std():.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = [('R²', scores_r2), ('MSE', -scores_mse), ('MAE', -scores_mae)]
colors = ['blue', 'green', 'orange']

for idx, (name, scores) in enumerate(metrics):
    ax = axes[idx]
    ax.bar(range(1, 6), scores, color=colors[idx], alpha=0.7, edgecolor='black')
    ax.axhline(scores.mean(), color='red', linestyle='--', linewidth=2, 
               label=f'Media: {scores.mean():.3f}')
    ax.set_xlabel('Fold', fontsize=12)
    ax.set_ylabel(name, fontsize=12)
    ax.set_title(f'{name} por Fold', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Ventajas y Desventajas de SVR

### Ventajas

1. **No linealidad**: Maneja relaciones no lineales mediante kernels
2. **Robustez**: Insensible a outliers gracias a la pérdida ε-insensitive
3. **Generalización**: Tiende a generalizar bien con parámetros adecuados

### Desventajas

1. **Escalabilidad**: No es adecuado para datasets muy grandes
2. **Sensibilidad a parámetros**: Requiere tuning cuidadoso
3. **Interpretabilidad**: Menos interpretable que modelos lineales

## 9. Buenas Prácticas

### Checklist para SVR

1. **Preprocesamiento**
   - Escalar características (StandardScaler)
   - Manejar valores atípicos

2. **Selección de Kernel**
   - Comenzar con RBF (más versátil)
   - Probar lineal para datos lineales

3. **Tuning de Hiperparámetros**
   - Usar GridSearchCV
   - Rango de C: [0.1, 1, 10, 100, 1000]
   - Rango de ε: [0.01, 0.1, 0.5, 1]

4. **Validación**
   - Validación cruzada (5-fold)
   - Evaluar en conjunto de prueba separado

## 10. Referencias

- [Scikit-learn SVR](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html)
- [Scikit-learn LinearSVR](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVR.html)